# 10. K-Means Model Preparation

Notebook ini bertugas **melatih dan menyimpan model K-Means Clustering** yang nantinya akan di-load oleh aplikasi Streamlit.

**Input:** `teams_style_scores.csv` — 9 Style Scores yang sudah dihasilkan dari tahap Style Scoring sebelumnya.

**Output:**
- `models/scaler.pkl` — StandardScaler
- `models/kmeans_k{2,3,4,5}.pkl` — Model K-Means untuk masing-masing K
- `clustering_results/clustering_k{2,3,4,5}.csv` — Hasil clustering untuk masing-masing K

**Pipeline:**
```
teams_style_scores.csv
        |
   9 Style Scores
        |
   StandardScaler
        |
      X_scaled
        |
 +------+------+------+------+
 | K=2  | K=3  | K=4  | K=5  |
 +------+------+------+------+
        |
  models/*.pkl  +  clustering_results/*.csv
```

> **Penting:** Streamlit hanya akan **load** model, bukan melatih ulang.

## Cell 1 — Import Libraries

In [14]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("Libraries loaded.")

Libraries loaded.


## Cell 2 — Configuration

Konfigurasi path, random state, dan daftar fitur yang digunakan untuk clustering.

In [15]:
# --- Path ---
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "gold" / "teams" / "teams_style_scores.csv"
MODEL_DIR = PROJECT_ROOT / "data" / "models"
RESULT_DIR = PROJECT_ROOT / "data" / "clustering_results"

# --- Hyperparameters ---
RANDOM_STATE = 42
N_INIT = 50

# --- Features ---
FEATURE_COLUMNS = [
    "possession_score",
    "attacking_efficiency_score",
    "direct_play_score",
    "pressing_score",
    "defensive_solidity_score",
    "chance_creation_score",
    "build_up_score",
    "set_piece_score",
    "counter_attack_score",
]

# --- Create directories ---
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_PATH    : {DATA_PATH}")
print(f"MODEL_DIR    : {MODEL_DIR}")
print(f"RESULT_DIR   : {RESULT_DIR}")
print(f"RANDOM_STATE : {RANDOM_STATE}")
print(f"N_INIT       : {N_INIT}")
print(f"Features     : {len(FEATURE_COLUMNS)} style scores")

PROJECT_ROOT : /Users/g/Desktop/Documents/LaLiga/laliga-project
DATA_PATH    : /Users/g/Desktop/Documents/LaLiga/laliga-project/data/gold/teams/teams_style_scores.csv
MODEL_DIR    : /Users/g/Desktop/Documents/LaLiga/laliga-project/data/models
RESULT_DIR   : /Users/g/Desktop/Documents/LaLiga/laliga-project/data/clustering_results
RANDOM_STATE : 42
N_INIT       : 50
Features     : 9 style scores


## Cell 3 — Load Data

Membaca `teams_style_scores.csv` — dataset ini berisi 1 kolom identifier (`club`) dan 9 kolom style scores.

In [16]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"Kolom: {list(df.columns)}")
print(f"Tipe data:\n{df.dtypes}")
print()
df.head()

Shape: (20, 10)
Kolom: ['club', 'possession_score', 'attacking_efficiency_score', 'direct_play_score', 'pressing_score', 'defensive_solidity_score', 'chance_creation_score', 'build_up_score', 'set_piece_score', 'counter_attack_score']
Tipe data:
club                              str
possession_score              float64
attacking_efficiency_score    float64
direct_play_score             float64
pressing_score                float64
defensive_solidity_score      float64
chance_creation_score         float64
build_up_score                float64
set_piece_score               float64
counter_attack_score          float64
dtype: object



,club,possession_score,attacking_efficiency_score,direct_play_score,pressing_score,defensive_solidity_score,chance_creation_score,build_up_score,set_piece_score,counter_attack_score
0,Alaves,35.54,28.84,42.39,42.00,58.01,29.60,29.93,55.21,27.11
1,Athletic Bilbao,35.48,25.37,50.96,77.96,64.27,46.15,31.02,34.02,38.08
2,Atlético Madrid,62.80,58.39,37.83,38.66,58.67,58.67,60.86,51.70,33.93
3,Barcelona,99.85,92.97,28.16,73.25,50.82,93.29,100.00,57.58,45.88
4,Celta Vigo,62.75,52.20,29.33,19.63,45.68,32.58,61.90,8.21,23.52


## Cell 4 — Validation

Memastikan dataset siap digunakan:
1. Semua `FEATURE_COLUMNS` tersedia
2. Tidak ada missing value
3. Semua fitur bertipe numerik
4. Tidak ada duplikat klub
5. Jumlah data sesuai

In [17]:
# 1. Cek ketersediaan kolom
missing_cols = set(FEATURE_COLUMNS) - set(df.columns)
assert len(missing_cols) == 0, f"Kolom tidak ditemukan: {missing_cols}"
print("[OK] Semua FEATURE_COLUMNS tersedia.")

# 2. Cek missing value
null_count = df[FEATURE_COLUMNS].isnull().sum().sum()
assert null_count == 0, f"Terdapat {null_count} missing values!"
print(f"[OK] Tidak ada missing value.")

# 3. Cek tipe data numerik
non_numeric = df[FEATURE_COLUMNS].select_dtypes(exclude=[np.number]).columns.tolist()
assert len(non_numeric) == 0, f"Kolom non-numerik: {non_numeric}"
print("[OK] Semua fitur bertipe numerik.")

# 4. Cek duplikat klub
dup_count = df["club"].duplicated().sum()
assert dup_count == 0, f"Terdapat {dup_count} duplikat klub!"
print(f"[OK] Tidak ada duplikat klub.")

# 5. Jumlah data
print(f"[OK] Jumlah klub: {len(df)}")
print()
print("Validasi selesai.")

[OK] Semua FEATURE_COLUMNS tersedia.
[OK] Tidak ada missing value.
[OK] Semua fitur bertipe numerik.
[OK] Tidak ada duplikat klub.
[OK] Jumlah klub: 20

Validasi selesai.


## Cell 5 — Prepare Features

Memisahkan identifier (`club`) dari fitur clustering (9 style scores).

In [18]:
clubs = df["club"].copy()
X = df[FEATURE_COLUMNS].copy()

print(f"X shape: {X.shape}")
print(f"Clubs  : {len(clubs)}")
print()
X.head()

X shape: (20, 9)
Clubs  : 20



,possession_score,attacking_efficiency_score,direct_play_score,pressing_score,defensive_solidity_score,chance_creation_score,build_up_score,set_piece_score,counter_attack_score
0,35.54,28.84,42.39,42.00,58.01,29.60,29.93,55.21,27.11
1,35.48,25.37,50.96,77.96,64.27,46.15,31.02,34.02,38.08
2,62.80,58.39,37.83,38.66,58.67,58.67,60.86,51.70,33.93
3,99.85,92.97,28.16,73.25,50.82,93.29,100.00,57.58,45.88
4,62.75,52.20,29.33,19.63,45.68,32.58,61.90,8.21,23.52


## Cell 6 — Standard Scaling

Menggunakan **satu** `StandardScaler` yang sama untuk seluruh model K-Means.

Scaler di-fit **sekali** di sini, kemudian disimpan ke `models/scaler.pkl`.

In [19]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Simpan scaler
scaler_path = MODEL_DIR / "scaler.pkl"
joblib.dump(scaler, scaler_path)

print(f"X_scaled shape: {X_scaled.shape}")
print(f"Mean (harus ~0): {X_scaled.mean(axis=0).round(6)}")
print(f"Std  (harus ~1): {X_scaled.std(axis=0).round(6)}")
print(f"\nScaler disimpan ke: {scaler_path}")

X_scaled shape: (20, 9)
Mean (harus ~0): [ 0. -0. -0.  0. -0.  0.  0.  0. -0.]
Std  (harus ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1.]

Scaler disimpan ke: /Users/g/Desktop/Documents/LaLiga/laliga-project/data/models/scaler.pkl


## Cell 7 — K-Means K=2

In [20]:
kmeans_k2 = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=N_INIT)
kmeans_k2.fit(X_scaled)

labels_k2 = kmeans_k2.labels_
inertia_k2 = kmeans_k2.inertia_
silhouette_k2 = silhouette_score(X_scaled, labels_k2)

result_k2 = pd.DataFrame({"club": clubs, "cluster": labels_k2})

# Simpan
joblib.dump(kmeans_k2, MODEL_DIR / "kmeans_k2.pkl")
result_k2.to_csv(RESULT_DIR / "clustering_k2.csv", index=False)

print(f"K=2  |  Inertia: {inertia_k2:.2f}  |  Silhouette: {silhouette_k2:.4f}")
print(f"\nCluster size:\n{result_k2['cluster'].value_counts().sort_index()}")
print()
result_k2.sort_values("club")

K=2  |  Inertia: 125.51  |  Silhouette: 0.4114

Cluster size:
cluster
0    18
1     2
Name: count, dtype: int64



,club,cluster
0,Alaves,0
1,Athletic Bilbao,0
2,Atlético Madrid,0
3,Barcelona,1
4,Celta Vigo,0
5,Elche,0
6,Espanyol,0
7,Getafe,0
8,Girona,0
9,Levante,0


## Cell 8 — K-Means K=3

In [21]:
kmeans_k3 = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=N_INIT)
kmeans_k3.fit(X_scaled)

labels_k3 = kmeans_k3.labels_
inertia_k3 = kmeans_k3.inertia_
silhouette_k3 = silhouette_score(X_scaled, labels_k3)

result_k3 = pd.DataFrame({"club": clubs, "cluster": labels_k3})

# Simpan
joblib.dump(kmeans_k3, MODEL_DIR / "kmeans_k3.pkl")
result_k3.to_csv(RESULT_DIR / "clustering_k3.csv", index=False)

print(f"K=3  |  Inertia: {inertia_k3:.2f}  |  Silhouette: {silhouette_k3:.4f}")
print(f"\nCluster size:\n{result_k3['cluster'].value_counts().sort_index()}")
print()
result_k3.sort_values("club")

K=3  |  Inertia: 91.06  |  Silhouette: 0.2537

Cluster size:
cluster
0    12
1     3
2     5
Name: count, dtype: int64



,club,cluster
0,Alaves,0
1,Athletic Bilbao,0
2,Atlético Madrid,1
3,Barcelona,1
4,Celta Vigo,2
5,Elche,2
6,Espanyol,0
7,Getafe,0
8,Girona,2
9,Levante,0


## Cell 9 — K-Means K=4

In [22]:
kmeans_k4 = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=N_INIT)
kmeans_k4.fit(X_scaled)

labels_k4 = kmeans_k4.labels_
inertia_k4 = kmeans_k4.inertia_
silhouette_k4 = silhouette_score(X_scaled, labels_k4)

result_k4 = pd.DataFrame({"club": clubs, "cluster": labels_k4})

# Simpan
joblib.dump(kmeans_k4, MODEL_DIR / "kmeans_k4.pkl")
result_k4.to_csv(RESULT_DIR / "clustering_k4.csv", index=False)

print(f"K=4  |  Inertia: {inertia_k4:.2f}  |  Silhouette: {silhouette_k4:.4f}")
print(f"\nCluster size:\n{result_k4['cluster'].value_counts().sort_index()}")
print()
result_k4.sort_values("club")

K=4  |  Inertia: 65.95  |  Silhouette: 0.2838

Cluster size:
cluster
0    6
1    3
2    9
3    2
Name: count, dtype: int64



,club,cluster
0,Alaves,2
1,Athletic Bilbao,2
2,Atlético Madrid,1
3,Barcelona,1
4,Celta Vigo,0
5,Elche,0
6,Espanyol,2
7,Getafe,2
8,Girona,0
9,Levante,2


## Cell 10 — K-Means K=5

In [23]:
kmeans_k5 = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=N_INIT)
kmeans_k5.fit(X_scaled)

labels_k5 = kmeans_k5.labels_
inertia_k5 = kmeans_k5.inertia_
silhouette_k5 = silhouette_score(X_scaled, labels_k5)

result_k5 = pd.DataFrame({"club": clubs, "cluster": labels_k5})

# Simpan
joblib.dump(kmeans_k5, MODEL_DIR / "kmeans_k5.pkl")
result_k5.to_csv(RESULT_DIR / "clustering_k5.csv", index=False)

print(f"K=5  |  Inertia: {inertia_k5:.2f}  |  Silhouette: {silhouette_k5:.4f}")
print(f"\nCluster size:\n{result_k5['cluster'].value_counts().sort_index()}")
print()
result_k5.sort_values("club")

K=5  |  Inertia: 50.39  |  Silhouette: 0.2780

Cluster size:
cluster
0    7
1    2
2    2
3    5
4    4
Name: count, dtype: int64



,club,cluster
0,Alaves,0
1,Athletic Bilbao,0
2,Atlético Madrid,0
3,Barcelona,2
4,Celta Vigo,3
5,Elche,3
6,Espanyol,4
7,Getafe,4
8,Girona,3
9,Levante,4


## Cell 11 — Summary

Ringkasan perbandingan seluruh model K-Means.

In [24]:
summary = pd.DataFrame({
    "K": [2, 3, 4, 5],
    "Inertia": [inertia_k2, inertia_k3, inertia_k4, inertia_k5],
    "Silhouette Score": [silhouette_k2, silhouette_k3, silhouette_k4, silhouette_k5],
})

print("=" * 50)
print("MODEL SUMMARY")
print("=" * 50)
print(summary.to_string(index=False))
print()

print("=" * 50)
print("CLUSTER SIZES")
print("=" * 50)
for k, result in [(2, result_k2), (3, result_k3), (4, result_k4), (5, result_k5)]:
    sizes = result["cluster"].value_counts().sort_index()
    size_str = "  ".join([f"C{i}:{n}" for i, n in sizes.items()])
    print(f"K={k}  ->  {size_str}")

MODEL SUMMARY
 K    Inertia  Silhouette Score
 2 125.511830          0.411415
 3  91.059801          0.253715
 4  65.952911          0.283794
 5  50.393420          0.278006

CLUSTER SIZES
K=2  ->  C0:18  C1:2
K=3  ->  C0:12  C1:3  C2:5
K=4  ->  C0:6  C1:3  C2:9  C3:2
K=5  ->  C0:7  C1:2  C2:2  C3:5  C4:4


## Cell 12 — Verify Exported Files

Memastikan semua file model dan hasil clustering berhasil disimpan.

In [25]:
expected_models = [
    MODEL_DIR / "scaler.pkl",
    MODEL_DIR / "kmeans_k2.pkl",
    MODEL_DIR / "kmeans_k3.pkl",
    MODEL_DIR / "kmeans_k4.pkl",
    MODEL_DIR / "kmeans_k5.pkl",
]

expected_results = [
    RESULT_DIR / "clustering_k2.csv",
    RESULT_DIR / "clustering_k3.csv",
    RESULT_DIR / "clustering_k4.csv",
    RESULT_DIR / "clustering_k5.csv",
]

print("=" * 50)
print("FILE VERIFICATION")
print("=" * 50)

all_ok = True
for f in expected_models + expected_results:
    exists = f.exists()
    size = f.stat().st_size if exists else 0
    status = f"OK ({size:,} bytes)" if exists else "MISSING"
    symbol = "[OK]" if exists else "[!!]"
    print(f"  {symbol} {f.name:30s} {status}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("Semua file berhasil disimpan.")
else:
    print("PERHATIAN: Ada file yang tidak ditemukan!")

FILE VERIFICATION
  [OK] scaler.pkl                     OK (1,231 bytes)
  [OK] kmeans_k2.pkl                  OK (951 bytes)
  [OK] kmeans_k3.pkl                  OK (1,031 bytes)
  [OK] kmeans_k4.pkl                  OK (1,095 bytes)
  [OK] kmeans_k5.pkl                  OK (1,175 bytes)
  [OK] clustering_k2.csv              OK (260 bytes)
  [OK] clustering_k3.csv              OK (260 bytes)
  [OK] clustering_k4.csv              OK (260 bytes)
  [OK] clustering_k5.csv              OK (260 bytes)

Semua file berhasil disimpan.


## Cell 13 — Verify Model Reload

Memuat kembali scaler dan seluruh model, kemudian memastikan hasil prediksi dari model yang di-load **identik** dengan hasil training sebelumnya.

Tahap ini **tidak melakukan fit ulang** — hanya `.predict()` menggunakan model yang sudah disimpan.

In [26]:
# Load scaler
scaler_loaded = joblib.load(MODEL_DIR / "scaler.pkl")
X_scaled_loaded = scaler_loaded.transform(X)

# Verifikasi scaler menghasilkan output yang sama
assert np.allclose(X_scaled, X_scaled_loaded), "Scaler output mismatch!"
print("[OK] Scaler: output identik.")

# Load dan verifikasi setiap model
original_labels = {
    2: labels_k2,
    3: labels_k3,
    4: labels_k4,
    5: labels_k5,
}

for k in [2, 3, 4, 5]:
    model_loaded = joblib.load(MODEL_DIR / f"kmeans_k{k}.pkl")
    pred_loaded = model_loaded.predict(X_scaled_loaded)
    match = np.array_equal(original_labels[k], pred_loaded)
    status = "[OK]" if match else "[!!]"
    print(f"{status} K={k}: prediksi {'identik' if match else 'BERBEDA'}")
    assert match, f"K={k} prediksi tidak cocok!"

print()
print("Semua model berhasil diverifikasi.")
print("Model siap digunakan oleh Streamlit.")

[OK] Scaler: output identik.
[OK] K=2: prediksi identik
[OK] K=3: prediksi identik
[OK] K=4: prediksi identik
[OK] K=5: prediksi identik

Semua model berhasil diverifikasi.
Model siap digunakan oleh Streamlit.
